In [8]:
# !pip install flask pandas python-dotenv openpyxl supabase
# !pip install supabase

In [9]:
# pip show supabase

In [10]:
import pandas as pd
import glob
import os
import io

In [50]:
ActivationDetailReport = pd.read_csv(r"D:\i\uploads\raw\ActivationDetailReport_7GF0SCN8M.csv",
    encoding="latin1")


In [12]:
CurCallidusReport.head()

,Unnamed: 0,marketid,custno,username,name,lastname,periodname,dealerpositionname,dealerfullname,dealerregionname,...,rma,originalsoc,compsoc,compmrc,custno1,fk_store,tradeindeducted,periodnamedate,callidusid,failedaudit
0,0,METRO-NY,C-70812973,BWH13560,Saurav,Prajapati,April 2026,8603383-NYC,The Portables Choice Group Llc,NaN,...,NaN,35GBNP,35GBNP,35.0,C-70812973,17433,NaN,2026-04-01,7821635573,NaN
1,1,METRO-NY,C-70812973,CGN92584,Carol,Aguayo,April 2026,8603383-NYC,The Portables Choice Group Llc,NaN,...,NaN,BFLX60,BFLX60,65.0,C-70812973,17433,NaN,2026-04-01,7821801635,NaN
2,2,METRO-NY,C-70812973,BWH13560,Saurav,Prajapati,April 2026,8603383-NYC,The Portables Choice Group Llc,NaN,...,NaN,TAB20SD,TAB20SD,20.0,C-70812973,17433,NaN,2026-04-01,7821949583,NaN
3,3,METRO-NY,C-70812973,BWH13560,Saurav,Prajapati,April 2026,8603383-NYC,The Portables Choice Group Llc,NaN,...,NaN,TAB20SD,TAB20SD,20.0,C-70812973,17433,NaN,2026-04-01,7823340408,NaN
4,4,METRO-NY,C-70812973,CGN92584,Carol,Aguayo,April 2026,8603383-NYC,The Portables Choice Group Llc,NaN,...,NaN,40L1WEB,40L1WEB,20.0,C-70812973,17433,NaN,2026-04-01,7823348679,NaN


In [13]:
CurCallidusReport.to_csv(
    r"D:\i\metro\curCallidus_7GF0SU2WW.csv"
)

In [15]:
# Collecting columns into new D_FMT

collecting_columns = ActivationDetailReport[['storeid','company','serial','accounttype','plancode','plantype1',]]

In [16]:
collecting_columns.head()

,storeid,company,serial,accounttype,plancode,plantype1
0,C-70812973,DKS WIRELESS 2128007-DCWP,'356795671974397,Prime,35GBNP,Voice
1,C-70812973,DKS WIRELESS 2128007-DCWP,'356795671974397,Prime,SCURE5S,Feature
2,C-70812973,DKS WIRELESS 2128007-DCWP,'016759005739068,Prime,TAB20SD,Voice
3,C-70812973,DKS WIRELESS 2128007-DCWP,'016759005737708,Prime,TAB20SD,Voice
4,C-70812973,DKS WIRELESS 2128007-DCWP,'864644063338388,Prime,HINTFOR,Voice


In [17]:
collecting_columns = collecting_columns[collecting_columns["plantype1"] == "Voice"]

In [34]:
collecting_columns['serial'] = (collecting_columns['serial'].astype(str).str.replace(r'\D', '', regex=True))

In [35]:
collecting_columns.head()

,storeid,company,serial,accounttype,plancode,plantype1,Compensation
0,C-70812973,DKS WIRELESS 2128007-DCWP,356795671974397,Prime,35GBNP,Voice,NaN
2,C-70812973,DKS WIRELESS 2128007-DCWP,016759005739068,Prime,TAB20SD,Voice,NaN
3,C-70812973,DKS WIRELESS 2128007-DCWP,016759005737708,Prime,TAB20SD,Voice,NaN
4,C-70812973,DKS WIRELESS 2128007-DCWP,864644063338388,Prime,HINTFOR,Voice,NaN
5,C-70812973,DKS WIRELESS 2128007-DCWP,350153153301919,Prime,4F100L1AP,Voice,NaN


In [18]:
# collecting_columns = collecting_columns.merge(
#     df3[['id', 'salary', 'department']],
#     on='serial',
#     how='left'
# )

In [36]:
serial_counts = collecting_columns['serial'].value_counts().reset_index()

In [37]:
serial_counts.columns = ['serial', 'count']

print(serial_counts)

              serial  count
0    356795671974397      1
1    016759005739068      1
2    016759005737708      1
3    864644063338388      1
4    350153153301919      1
..               ...    ...
215  358926211250554      1
216  350321564356703      1
217  861035062868892      1
218  350321564356539      1
219  864644062918719      1

[220 rows x 2 columns]


In [38]:
duplicates = serial_counts[serial_counts['count'] > 1]

print(duplicates)

Empty DataFrame
Columns: [serial, count]
Index: []


this shows that there is no duplicate serial number present in ActivationDetailReport

is it correct statement that  ActivationDetailReport will never have duplicates.

now moving towards curCallidus report we need esn=serial for maping

In [39]:
CurCallidusReport = pd.read_csv(r"D:\i\uploads\raw\curCallidus_7GF0SU2WW.csv",
    encoding="latin1")

In [40]:
CurCallidusReport.head()

,Unnamed: 0,marketid,custno,username,name,lastname,periodname,dealerpositionname,dealerfullname,dealerregionname,...,rma,originalsoc,compsoc,compmrc,custno1,fk_store,tradeindeducted,periodnamedate,callidusid,failedaudit
0,0,METRO-NY,C-70812973,BWH13560,Saurav,Prajapati,April 2026,8603383-NYC,The Portables Choice Group Llc,NaN,...,NaN,35GBNP,35GBNP,35.0,C-70812973,17433,NaN,2026-04-01,7821635573,NaN
1,1,METRO-NY,C-70812973,CGN92584,Carol,Aguayo,April 2026,8603383-NYC,The Portables Choice Group Llc,NaN,...,NaN,BFLX60,BFLX60,65.0,C-70812973,17433,NaN,2026-04-01,7821801635,NaN
2,2,METRO-NY,C-70812973,BWH13560,Saurav,Prajapati,April 2026,8603383-NYC,The Portables Choice Group Llc,NaN,...,NaN,TAB20SD,TAB20SD,20.0,C-70812973,17433,NaN,2026-04-01,7821949583,NaN
3,3,METRO-NY,C-70812973,BWH13560,Saurav,Prajapati,April 2026,8603383-NYC,The Portables Choice Group Llc,NaN,...,NaN,TAB20SD,TAB20SD,20.0,C-70812973,17433,NaN,2026-04-01,7823340408,NaN
4,4,METRO-NY,C-70812973,CGN92584,Carol,Aguayo,April 2026,8603383-NYC,The Portables Choice Group Llc,NaN,...,NaN,40L1WEB,40L1WEB,20.0,C-70812973,17433,NaN,2026-04-01,7823348679,NaN


In [41]:
transaction_per_esn = CurCallidusReport.groupby('esn', as_index=False)['transaction_amount'].sum()

print(transaction_per_esn)

                 esn  transaction_amount
0                  0               275.0
1     15249003708225                25.0
2     16601007162552                25.0
3     16749000728125                21.4
4     16759005737708                11.0
..               ...                 ...
237  864644069560415                 0.0
238  866080076694137               137.5
239  866134075318238               300.0
240  867237078212321                20.0
241  999999999999999              -372.5

[242 rows x 2 columns]


       esn  transaction_amount
0                  0               275.0
1     15249003708225                25.0
2     16601007162552                25.0
3     16749000728125                21.4
why these many 0 in esn


In [45]:
# count common entries between serial and esn

common_count = collecting_columns['serial'].isin(
    transaction_per_esn['esn']
).sum()

print("Common entries:", common_count)

Common entries: 0


In [47]:
# To inspect matched serials:

common_values = set(collecting_columns['serial']).intersection(
    set(transaction_per_esn['esn'])
)

print(common_values)
print("Total common unique values:", len(common_values))

set()
Total common unique values: 0


In [48]:
# To inspect unmatched serials:

unmatched = collecting_columns[
    ~collecting_columns['serial'].isin(transaction_per_esn['esn'])
]

print(unmatched[['serial']].head())

            serial
0  356795671974397
2  016759005739068
3  016759005737708
4  864644063338388
5  350153153301919


In [42]:
# match serial with esn and bring transaction_amount into new column Compensation

collecting_columns['Compensation'] = (
    collecting_columns['serial']
    .map(transaction_per_esn.set_index('esn')['transaction_amount'])
)

In [43]:
collecting_columns.head()

,storeid,company,serial,accounttype,plancode,plantype1,Compensation
0,C-70812973,DKS WIRELESS 2128007-DCWP,356795671974397,Prime,35GBNP,Voice,NaN
2,C-70812973,DKS WIRELESS 2128007-DCWP,016759005739068,Prime,TAB20SD,Voice,NaN
3,C-70812973,DKS WIRELESS 2128007-DCWP,016759005737708,Prime,TAB20SD,Voice,NaN
4,C-70812973,DKS WIRELESS 2128007-DCWP,864644063338388,Prime,HINTFOR,Voice,NaN
5,C-70812973,DKS WIRELESS 2128007-DCWP,350153153301919,Prime,4F100L1AP,Voice,NaN


In [49]:
# check unique values in Compensation column
print(collecting_columns['Compensation'].unique())

[nan]


Now moving towards callidus report file 

In [51]:
CallidusReport = pd.read_csv(r"D:\i\uploads\raw\CallidusDetail_7GF0T17DL.csv",
    encoding="latin1")

In [53]:
CallidusReport.info()

<class 'pandas.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 45 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   marketid                 117 non-null    str    
 1   custno                   117 non-null    str    
 2   username                 117 non-null    str    
 3   name                     117 non-null    str    
 4   lastname                 117 non-null    str    
 5   periodseq                0 non-null      float64
 6   periodname               117 non-null    str    
 7   dealerpositionname       117 non-null    str    
 8   dealerfullname           117 non-null    str    
 9   dealerpayeeid            117 non-null    str    
 10  dealerparticipantseq     0 non-null      float64
 11  dealerpositionseq        0 non-null      float64
 12  dealerregionname         0 non-null      float64
 13  dealeraddress            0 non-null      float64
 14  doorpositionname         117 non-null

In [55]:
# map transaction_amount into new column Rebate
collecting_columns['Rebate'] = (
    collecting_columns['serial']
    .map(CallidusReport.set_index('esn')['transaction_amount'])
)

In [56]:
collecting_columns.head()

,storeid,company,serial,accounttype,plancode,plantype1,Compensation,Rebate
0,C-70812973,DKS WIRELESS 2128007-DCWP,356795671974397,Prime,35GBNP,Voice,NaN,NaN
2,C-70812973,DKS WIRELESS 2128007-DCWP,016759005739068,Prime,TAB20SD,Voice,NaN,NaN
3,C-70812973,DKS WIRELESS 2128007-DCWP,016759005737708,Prime,TAB20SD,Voice,NaN,NaN
4,C-70812973,DKS WIRELESS 2128007-DCWP,864644063338388,Prime,HINTFOR,Voice,NaN,NaN
5,C-70812973,DKS WIRELESS 2128007-DCWP,350153153301919,Prime,4F100L1AP,Voice,NaN,NaN
